# Notebook 02 — Preprocessing Pipeline

Goals:
- Visualise the effect of brain windowing on HU values
- Confirm resize correctness for volume and mask
- Validate the PyTorch DataLoader output shape and value range
- Show before/after preprocessing side-by-side

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from src.data.loader import load_nifti, build_index, train_val_split
from src.preprocessing.transforms import (
    apply_brain_window, normalize, resize_volume, resize_mask,
    preprocess, BRAIN_HU_MIN, BRAIN_HU_MAX, TARGET_SHAPE
)
from src.data.dataset import BrainCTDataset
from src.visualization.plots import show_slices

DATA_ROOT = '../data/raw'
print('Setup OK')

## 1. Brain windowing — before and after

We use the **brain soft-tissue window**: [-5, 75] HU.

This clips bone (>700 HU) and air (~-1000 HU), leaving only the soft-tissue range where pathology lives.
The result is then min-max normalized to [0, 1] for the model.

In [ ]:
records = build_index(DATA_ROOT)
volume_raw, spacing = load_nifti(records[0]['image'])
mask_raw, _         = load_nifti(records[0]['mask'])
mask_raw = (mask_raw > 0.5).astype(np.uint8)

d = volume_raw.shape[0] // 2

volume_windowed = apply_brain_window(volume_raw)
volume_normed   = normalize(volume_raw)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(volume_raw[d], cmap='gray', origin='lower')
axes[0].set_title(f'Raw CT\nHU range: [{volume_raw.min():.0f}, {volume_raw.max():.0f}]')
axes[0].axis('off')

axes[1].imshow(volume_windowed[d], cmap='gray', origin='lower',
               vmin=BRAIN_HU_MIN, vmax=BRAIN_HU_MAX)
axes[1].set_title(f'After brain window\n[{BRAIN_HU_MIN}, {BRAIN_HU_MAX}] HU')
axes[1].axis('off')

axes[2].imshow(volume_normed[d], cmap='gray', origin='lower', vmin=0, vmax=1)
axes[2].set_title(f'Normalized to [0, 1]\nmean={volume_normed.mean():.3f}')
axes[2].axis('off')

plt.suptitle('Preprocessing: windowing + normalization (axial center slice)', fontsize=13)
plt.tight_layout()
plt.show()

## 2. Resize — volume and mask

- **Volume**: trilinear interpolation (`order=1`) — smooth, preserves intensity
- **Mask**: nearest-neighbor (`order=0`) — critical! Linear interpolation would create fractional boundary values (e.g. 0.3) that corrupt the binary label

In [ ]:
vol_pp, mask_pp = preprocess(volume_raw, mask_raw)

print('Original → Preprocessed')
print(f'  Volume shape : {volume_raw.shape} → {vol_pp.shape}')
print(f'  Mask shape   : {mask_raw.shape} → {mask_pp.shape}')
print(f'  Volume range : [{volume_raw.min():.1f}, {volume_raw.max():.1f}] → [{vol_pp.min():.3f}, {vol_pp.max():.3f}]')
print(f'  Mask values  : {np.unique(mask_raw)} → {np.unique(mask_pp)} (must be {{0, 1}})')
print(f'  Lesion voxels: {mask_raw.sum()} → {mask_pp.sum()}')

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for i, (vol, msk, label) in enumerate([
    (volume_raw, mask_raw, f'Original {volume_raw.shape}'),
    (vol_pp, mask_pp, f'Preprocessed {vol_pp.shape}'),
]):
    d = vol.shape[0] // 2
    for j, (img, title) in enumerate([
        (vol[d],  'CT slice'),
        (msk[d],  'Mask'),
        (vol[d],  'Overlay'),
    ]):
        ax = axes[i][j]
        ax.imshow(img, cmap='gray', origin='lower')
        if j == 2 and msk[d].any():
            overlay = np.zeros((*msk[d].shape, 4), dtype=np.float32)
            overlay[msk[d] > 0] = [1, 0, 0, 0.4]
            ax.imshow(overlay, origin='lower')
        ax.set_title(f'{label} — {title}')
        ax.axis('off')

plt.tight_layout()
plt.show()

## 3. PyTorch DataLoader validation

Confirm the DataLoader returns the expected shape and value range.

In [ ]:
train_records, val_records = train_val_split(records, val_fraction=0.2)

train_ds = BrainCTDataset(train_records[:8], augment=True)
val_ds   = BrainCTDataset(val_records[:4],  augment=False)

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=2, shuffle=False)

x_batch, y_batch = next(iter(train_loader))

print('Batch shapes')
print(f'  x : {x_batch.shape}  dtype={x_batch.dtype}')
print(f'  y : {y_batch.shape}  dtype={y_batch.dtype}')
print()
print('Value ranges')
print(f'  x min={x_batch.min():.4f}  max={x_batch.max():.4f}  (should be [0, 1])')
print(f'  y unique values: {y_batch.unique().tolist()}  (should be [0.0, 1.0])')
print()
print('No NaN/Inf in x:', torch.isfinite(x_batch).all().item())